## What are these CSDS files?

PureSkill.gg extracts the data from each CS2 demo into outputs called **channels**. Each channel is one of four types:

- Telemetry: information over time, such as each player's position on every tick (`player_vector`).
- Single event: one kind of event, such as `player_death`.
- Multi event: several events of a similar kind grouped together, such as the bomb events.
- Header: facts about the match, such as `map_name` and `tick_rate`.

A match has 42 channels, all compressed parquet files, plus the `csds` index object, which lists every channel and its columns. You rarely need every channel, so the readers accept "instructions" that name the channels and columns to read. The [CSDS spec](https://docs.pureskill.gg/datascience/adx/cs2/csds/spec) documents every channel and column.

Each demo goes through our parser and then a post-parser processor (PPP), which fixes some things and extends some channels with information from others. For instance, most events carry a `player_id`, but that number can change when a player disconnects and reconnects. So we add `player_id_fixed`, which stays the same for a player for the whole match.

## The sample matches

This repo comes with four CS2 matches from 2026-10-09 in its `sample_data` folder (its README lists them): two Premier matches on Valve servers and two FACEIT matches, about 30 MB in all.

They come from the [PureSkill.gg Competitive CS2 Gameplay](https://docs.pureskill.gg/datascience/adx/cs2/csds/) data set, licensed [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): no commercial use, attribute PureSkill.gg, and share anything derived from it under the same license. Anything you publish from the data must carry the text "Data provided by PureSkill.gg."

For more matches, subscribe to the data set on the AWS Data Exchange; [notebook 7](7%20-%20Getting%20csds%20data%20from%20the%20ADX.ipynb) shows how. Keep the `csds/yyyy/mm/dd/<match>/` folder layout of any data you add: each match's index names its files by that path.

_**Run this notebook as-is.**_

In [ ]:
from pureskillgg_makenew_pyskill.notebook import setup_notebook

In [ ]:
setup_notebook()

In [ ]:
import gzip
import json
import os
from pathlib import Path

ds_collection_path = Path(os.environ['PURESKILLGG_TOME_DS_COLLECTION_PATH'])
indexes = sorted(p for p in ds_collection_path.rglob('csds') if p.is_file())
print(len(indexes), 'matches in', ds_collection_path)
for index in indexes:
    print(' ', index.parent.relative_to(ds_collection_path).as_posix())

In [ ]:
# Read one match's index object (gzipped JSON) and list a few of its channels
raw = indexes[0].read_bytes()
manifest = json.loads(gzip.decompress(raw) if raw[:2] == b'\x1f\x8b' else raw)
print(len(manifest['channels']), 'channels, for example:')
for channel in manifest['channels'][:8]:
    print(' ', channel['channel'], '-', len(channel['columns']), 'columns')

Advance to the [next notebook](3%20-%20Make%20header%20tome.ipynb).